##  Install Required Libraries

In [ ]:
!pip install pandas numpy matplotlib seaborn scikit-learn xgboost lightgbm shap lime plotly streamlit -q

## Import Required Libraries

In [ ]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

from sklearn.preprocessing import MinMaxScaler, LabelEncoder
from sklearn.model_selection import train_test_split

from sklearn.neighbors import NearestNeighbors
from sklearn.metrics.pairwise import cosine_similarity

from sklearn.cluster import KMeans, DBSCAN
from sklearn.decomposition import PCA

from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, roc_auc_score, confusion_matrix, classification_report)


from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier

import warnings
warnings.filterwarnings("ignore")

import os 
import shap
import joblib

print('All libraries imported successfully!')

In [ ]:
df=pd.read_csv("data/dataset.csv")
df.shape
df.head()

In [ ]:
df.isnull().sum()
print("duplicated values:",df.duplicated().sum())

In [ ]:
df=df.dropna()
df=df.drop_duplicates()

In [ ]:
corr_matrix= df.corr(numeric_only=True)
plt.figure(figsize=(10,8))
sns.heatmap(corr_matrix, annot=True, fmt='.2f', cmap="YlGnBu")
plt.show()

## Feature engineering and encoding

In [ ]:
#  Popularity normalization 
df['popularity'] = df['popularity'] / 100

#  Duration in minutes 
df['duration_mins'] = df["duration_ms"] / 60000

# Encoded explicit column (true=1 | false=0)
df['explicit'] = df['explicit'].astype(int)

# Encoded track_genre 
le=LabelEncoder()
df["encoded_genre"] = le.fit_transform(df['track_genre'])

# Mood score: average of valence and energy
df["mood_score"] = (df["valence"] + df["energy"]) / 2

# Hit label: 1 if popularity>=0.75 else 0
df["hit"] = (df["popularity"] >= 0.75).astype(int)

print("Features engineered and encoded")


## Scaling the features


In [ ]:
#Scale the audio features
features = ['danceability','energy','loudness','speechiness','acousticness','instrumentalness','liveness','valence','tempo','duration_mins']
scaler=MinMaxScaler()

# Make a copy of df 
df_scaled=df.copy()
df_scaled[features]=scaler.fit_transform(df[features])

## Save the clean data

In [ ]:
os.makedirs('data', exist_ok=True)
df.to_csv('data/cleaned_spotify', index=False)
print("new csv created")

## Exploratory Data Analysis

In [ ]:
# Top 20 genres by average popularity
top_genres=df.groupby('track_genre')['popularity'].mean().sort_values(ascending=False).head(20)
fig = px.bar(x=top_genres.index, y=top_genres.values,
    title='Top 20 Genres by Average Popularity',
    labels={'x': 'Genre', 'y': 'Average Popularity'},
    color=top_genres.values, 
    color_continuous_scale='Viridis')
fig.show()
plt.show()



In [ ]:
df[features].hist(bins=20, figsize=(15,10))
plt.tight_layout()
plt.show()

In [ ]:
# Correlation heatmap 
corr_features = features + ["popularity"]
corr=df[corr_features].corr()
plt.figure(figsize=(10,8))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='YlGnBu')
plt.tight_layout()
plt.show()

---
## HIT PREDICTION

In [ ]:

X= df[features]
y= df['hit']

print("Hit songs:", y.sum())
print("Non-hit songs:", len(y) - y.sum())

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print("Data split into training and testing sets")

print("test set shape:", X_test.shape)
print("training set shape:", X_train.shape)
print('Class ratio:', round(y.mean() * 100, 2), '%')


## Train multiple models

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),   
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state= 42),    
    "XGBoost": XGBClassifier(n_estimators=100, random_state=42, use_label_encoder=False, eval_metric='logloss'),
    "LightGBM": LGBMClassifier(n_estimators=100, random_state=42)
}
results = []
for name, model in models.items():
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]

    results.append({
        "Model": name,
        "Accuracy": round(accuracy_score(y_test, y_pred), 4),
        "Precision": round(precision_score(y_test, y_pred), 4),
        "Recall": round(recall_score(y_test, y_pred), 4),
        "F1 Score": round(f1_score(y_test, y_pred), 4),
        "ROC AUC": round(roc_auc_score(y_test, y_prob), 4)
    })
results_df = pd.DataFrame(results).sort_values(by="ROC AUC", ascending=False)
results_df

    

In [ ]:
best_model=models['Random Forest']
y_pred=best_model.predict(X_test)  
print("Classification Report for Random Forest:")
print(classification_report(y_test, y_pred, target_names=['Non-Hit', 'Hit']))   


In [ ]:

print("Confusion Matrix for Random Forest:")
cm = confusion_matrix(y_test, y_pred)
cm_df = pd.DataFrame(cm, index=['Actual Non-Hit', 'Actual Hit'], columns=['Predicted Non-Hit', 'Predicted Hit'])
cm_df

In [ ]:
## Feature importance for Random Forest
importances = pd.DataFrame(best_model.feature_importances_, index=features, columns=['Importance'])
importances


In [ ]:
# TRY Balanced Random Forest to improve recall for "hits"

rf_balanced=RandomForestClassifier(n_estimators=100, random_state=42,class_weight='balanced')
rf_balanced.fit(X_train, y_train) 
y_pred_balanced=rf_balanced.predict(X_test)
print("Classification Report for Balanced Random Forest:")
print(classification_report(y_test, y_pred_balanced, target_names=['Non-Hit', 'Hit']))

In [ ]:
# TRY custom threshold to improve recall for "hits"

y_prob = best_model.predict_proba(X_test)[:,1]
y_pred_custom=(y_prob >= 0.3).astype(int)
print("Classification Report for Custom Threshold (0.3) Random Forest:")
print(classification_report(y_test, y_pred_custom, target_names=['Non-Hit', 'Hit']))

In [ ]:
# SHAP
explainer = shap.TreeExplainer(best_model)
X_sample = X_test.sample(500, random_state=42)
shap_values = explainer.shap_values(X_sample)
shap.summary_plot(shap_values, X_sample, feature_names=features)


## Save the model

In [ ]:
os.makedirs('models', exist_ok=True)
joblib.dump(best_model, 'models/hit_predictor.pkl')
joblib.dump(scaler, 'models/scaler.pkl')
joblib.dump(X.columns.tolist(), 'models/feature_columns.pkl')
print("dumped the pkl files")

# PART-2: MUSIC RECOMMENDATION SYSTEM

In [ ]:
## Select recommendation features 
rec_features = ['danceability','energy','loudness','speechiness','acousticness','instrumentalness','valence','tempo']

# Create recommendation dataframe
df_rec=df[rec_features + ['track_name','artists','track_genre','popularity']].reset_index(drop=True)
df_rec.head()

rec_scaler=MinMaxScaler()
X_recommended = rec_scaler.fit_transform(df_rec[rec_features])
X_recommended.shape

## Recommend Top similar songs

In [ ]:
def recommend_songs(song_name, top_n):

    # Find matching song
    matches = df_rec[df_rec['track_name'].str.lower() == song_name.lower()]

    if matches.empty:
        print(f'Song "{song_name}" not found.')
        return None

    # Get song index
    idx = matches.index[0]

    # Get feature vector
    song_vector = X_recommended[idx].reshape(1, -1)

    # Calculate cosine similarity
    sim_scores = cosine_similarity(song_vector, X_recommended)[0]

    # Create temporary dataframe with similarity scores
    temp_df = df_rec.copy()

    temp_df['similarity_score'] = sim_scores

    # Remove the input song itself
    temp_df = temp_df[temp_df.index != idx]

    # Keep only songs with popularity >= 0.75
    temp_df = temp_df[temp_df['popularity'] >= 0.75]

    # Sort by similarity score
    temp_df = temp_df.sort_values(by='similarity_score', ascending=False) 

    # Remove repeated song names
    temp_df = temp_df.drop_duplicates(subset='track_name')

    # Select top N songs
    result = temp_df[['track_name', 'artists', 'track_genre',
                      'popularity', 'similarity_score']].head(top_n)

    # Reset index
    result = result.reset_index(drop=True)

    return result


# Example
recommend_songs('smooth criminal', 10)


## Top songs in a genre

In [ ]:
def recommend_genres(genre, top_n):
    # Filter songs of the specified genre
    genre_songs = df_rec[df_rec['track_genre'].str.lower() == genre.lower()]
    if genre_songs.empty:
        print(f'Genre "{genre}" not found.')
        return None
    
    top_songs = genre_songs.sort_values(by='popularity', ascending=False).head(top_n)
    return top_songs[['track_name', 'artists', 'popularity']].reset_index(drop=True)

# Example
recommend_genres('pop', 20)


## Mood-Based Playlist generator   

In [ ]:
def mood_based(mood_type, top_n):
    rules = {
        'workout': (df['energy'] > 0.7) & (df['tempo'] > 0.6),
        'study':   (df['instrumentalness'] > 0.4) & (df['energy'] < 0.5),
        'relax':   (df['acousticness'] > 0.6) & (df['energy'] < 0.4),
        'party':   (df['danceability'] > 0.7) & (df['energy'] > 0.6),
        'happy':   (df['valence'] > 0.7) & (df['energy'] > 0.5),
        'sad':     (df['valence'] < 0.3) & (df['energy'] < 0.5),
    }
    if mood_type not in rules:
        print(f'Mood type "{mood_type}" not recognized. Choose from: {list(rules.keys())}')
        return None
    filtered_songs = df[rules[mood_type]]
    top_songs = filtered_songs.sort_values(by='popularity', ascending=False).drop_duplicates(subset='track_name').head(top_n)

    return top_songs[['track_name', 'artists', 'popularity']].reset_index(drop=True)  
            

# Example   
mood_based('party', 10)

# Phase-4 Clustering the songs

In [ ]:
audio_features = ['danceability', 'energy', 'valence', 'acousticness','tempo', 'loudness', 'speechiness', 'instrumentalness']

# sample 20000 songs
df_cluster = df.sample(20000, random_state=42).reset_index(drop=True)
cluster_scaler = MinMaxScaler()
X_cluster = cluster_scaler.fit_transform(df_cluster[audio_features])
X_cluster.shape


In [ ]:
# Finding Optimal K using Elbow method
inertia = []
K_range = range(1, 11)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42)
    km.fit(X_cluster)
    inertia.append(km.inertia_)
plt.figure(figsize=(8,5))
plt.plot(K_range, inertia, marker='o')
plt.title('Elbow Method for Optimal K') 
plt.xlabel('Number of Clusters (K)')
plt.ylabel('Inertia')
plt.tight_layout()
plt.show()

In [ ]:
# K-Means clustering
k=5
kmeans = KMeans(n_clusters=k, random_state=42)
df_cluster['kmeans_cluster'] = kmeans.fit_predict(X_cluster)
df_cluster.head()
cluster_names = {
    0: 'Party Songs',
    1: 'Workout Songs',
    2: 'Calm Acoustic',
    3: 'Emotional Songs',
    4: 'Relaxing Songs'
}
df_cluster['cluster_name'] = df_cluster['kmeans_cluster'].map(cluster_names)
df_cluster[['track_name', 'artists', 'cluster_name']].head(10)
df_cluster['cluster_name'].value_counts()

In [ ]:
# PCA Visualization
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_cluster)
plt.figure(figsize=(8,6))

figure = px.scatter(df_cluster, x=X_pca[:, 0], y=X_pca[:, 1], 
    color='cluster_name',
    title='PCA Visualization of Clusters',
    labels={'x': 'First Principal Component', 'y': 'Second Principal Component'},
    hover_data=['track_name', 'artists'] )

figure.show()
 